# Conversational Search Retrieval Augmented Generation

In this notebook you will implement the following steps:

- **Answer selection + evaluation**: Implement a *search-based* conversation framework evaluation framework to evaluate conversation topics made up of conversation turns.
- **Answer ranking**: Implement a *re-ranking method* to sort the initial search results. Evaluate the re-ranked results.
- **Conversation memory**: Implement a conversational context modeling method to keep track of the conversation state. 

Submission dates:
- **20 October**: first stage retrieval + conversation memory + evaluation
- **15 November**: re-ranking with LLM + evaluation
- **15 December**: answer generation + evaluation

## Test bed and conversation topics
The TREC CAST corpus (http://www.treccast.ai/) for Conversational Search is indexed in this cluster and available to be searched behind an OpenSearch API.

The queries and the relevance judgments are available through class `ConvSearchEvaluation`:

In [ ]:
from importlib import reload
import TRECCASTeval as trec
import numpy as np
import pprint

import numpy as np

pp = pprint.PrettyPrinter(indent=4)

test_bed = trec.ConvSearchEvaluation()

print()
print("========================================== Training conversations =====")
topics = {}
for topic in test_bed.train_topics:
    conv_id = topic['number']

    if conv_id not in (1, 2, 4, 7, 15, 17,18,22,23,24,25,27,30):
        continue

    print()
    print(conv_id, "  ", topic['title'])

    for turn in topic['turn']:
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)
        
        print(topic_turn_id, utterance)
        topics[topic_turn_id] = utterance

print()
print("========================================== Test conversations =====")
for topic in test_bed.test_topics:
    conv_id = topic['number']

    if conv_id not in (31, 32, 33, 34, 37, 40, 49, 50, 54, 56, 58, 59, 61, 67, 68, 69, 75, 77, 78, 79):
        continue

    print()
    print(conv_id, "  ", topic['title'])

    for turn in topic['turn']:
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)
        
        print(topic_turn_id, utterance)
        topics[topic_turn_id] = utterance


## OpenSearch

In [2]:
import OpenSearchSimpleAPI as osearch
import pprint as pp
opensearch = osearch.OSsimpleAPI()

Search example:

In [ ]:
results = opensearch.search_body(topics['33_1'], numDocs = 10)
print(results)

In [ ]:
opensearch.doc_term_vectors('MARCO_34196')

In [ ]:
# sum_doc_freq: 
# doc_count: 
# sum_ttf: 
# doc_freq: 
# ttf: 
# term_freq: 
opensearch.termvectors_JSON(doc_id='MARCO_34196')

In [ ]:
opensearch.get_doc_body('MARCO_34196')

In [ ]:
example_doc = 'The film only adapts the first half of the book, and consequently does not convey the message of the title as it was portrayed in the novel. The second half of the book would subsequently be used as the rough basis for the second film, The NeverEnding Story II: The Next Chapter.The third film, The NeverEnding Story III: Escape From Fantasia, features a completely original plot.he second half of the book would subsequently be used as the rough basis for the second film, The NeverEnding Story II: The Next Chapter.'
opensearch.query_terms(example_doc,'standard')

In [ ]:
opensearch.analyzer(analyzer="standard", query=example_doc)

## Retrieval with training conversations
The OpenSearchSimpleAPI notebook illustrates how to use OpenSearch. Use this API to retrieve the top 100 ranked passages for each conversation turn. 

To evaluate the results you should use the provided `ConvSearchEvaluation` class. Examine and discuss the recall metric results. In terms of metrics, discuss what should be your goals for each step of the project.

In [4]:
from importlib import reload
import TRECCASTeval as trec
from textwrap import wrap
import OpenSearchSimpleAPI as osearch
import rank_with_history as rwh
import ReRank as rr

import pprint as pp

osearch = reload(osearch)
trec = reload(trec)
rwh = rwh.rank_with_history()
rr = rr.ReRank()

opensearch = osearch.OSsimpleAPI()
test_bed = trec.ConvSearchEvaluation()
conv_topics = test_bed.train_topics
selected_topics = (1, 2, 4, 7, 17,18,22,23,24,25,27,30)

BM25

In [ ]:
from retrieval.BM25 import BM25

bm25 = BM25(opensearch,test_bed,conv_topics,selected_topics)
bm25.run()


LMD

In [ ]:
from retrieval.LMD import LMD

lmd = LMD(opensearch,test_bed,conv_topics,selected_topics)
lmd.run()

Conversation History - Sliding Window

In [ ]:
from retrieval.ConversationHistory import ConversationHistory

convHistory = ConversationHistory(opensearch,test_bed,conv_topics,selected_topics)
convHistory.run_sliding_window()

Conversation History - First utterance only

In [ ]:
from retrieval.ConversationHistory import ConversationHistory

convHistory = ConversationHistory(opensearch,test_bed,conv_topics,selected_topics)
convHistory.run_first_utt_only()

## Retrieval with the test conversations

In [9]:
from importlib import reload
import TRECCASTeval as trec
from textwrap import wrap
import OpenSearchSimpleAPI as osearch
import rank_with_history as rwh
import ReRank as rr

import pprint as pp

osearch = reload(osearch)
trec = reload(trec)
rwh = rwh.rank_with_history()
rr = rr.ReRank()

opensearch = osearch.OSsimpleAPI()

test_bed = trec.ConvSearchEvaluation()
conv_topics = test_bed.test_topics
verbose = False
selected_topics = (31, 32, 33, 34, 37, 40, 49, 50, 54, 56, 58, 59, 61, 67, 68, 69, 75, 77, 78, 79)

BM25

In [ ]:
from retrieval.BM25 import BM25

bm25 = BM25(opensearch,test_bed,conv_topics,selected_topics)
bm25.run()


LMD

In [ ]:
from retrieval.LMD import LMD

lmd = LMD(opensearch,test_bed,conv_topics,selected_topics)
lmd.run()

Conversation History - Sliding Window

In [ ]:
from retrieval.ConversationHistory import ConversationHistory

convHistory = ConversationHistory(opensearch,test_bed,conv_topics,selected_topics)
convHistory.run_sliding_window()

Conversation History - First utterance only

In [ ]:
from retrieval.ConversationHistory import ConversationHistory

convHistory = ConversationHistory(opensearch,test_bed,conv_topics,selected_topics)
convHistory.run_first_utt_only()